In [ ]:
import pandas as pd
import numpy as np
import os
import json
from dotenv import load_dotenv
import requests
import time

In [ ]:
import sys
from pathlib import Path

for candidato in (Path.cwd(), *Path.cwd().resolve().parents):
    if (candidato / "src").is_dir():
        if str(candidato) not in sys.path:
            sys.path.insert(0, str(candidato))
        break

from src.paths import RAIZ, DIR_RAW, DIR_PROCESSED, DIR_REPORTS

PATH_AEMET_CRUDO = DIR_RAW / "aemet_diario_8estaciones_2023_2026.parquet"
PATH_CUARENTENA = DIR_RAW / "cuarentena_aemet.json"

In [ ]:
# Carga las variables del archivo .env en el entorno del sistema
# Si el archivo .env está en la misma carpeta que este notebook, basta con esto:
# Al usar '../.env', le decimos que salga de 'sandbox' y busque el archivo en la raíz
load_dotenv()
# Si tu .env está en una carpeta superior o una ruta específica, puedes indicarla:
# load_dotenv(dotenv_path="../.env")

# Recuperamos el token de forma segura
TOKEN = os.getenv("API_AEMET")

# Verificación rápida (sin mostrar el token entero por seguridad)
if TOKEN:
    print(f"✅ Token cargado correctamente. Longitud: {len(TOKEN)} caracteres.")
else:
    print("❌ No se pudo encontrar la variable ESIOS_TOKEN. Revisa la ruta del archivo .env.")

In [ ]:
# --- PASO 1: Endpoint REAL y VIVO (Inventario de estaciones climatológicas) ---
# Este endpoint no caduca, es la entrada oficial
url = "https://opendata.aemet.es/opendata/api/valores/climatologicos/inventarioestaciones/todasestaciones/"

querystring = {"api_key": TOKEN}
headers = {'cache-control': "no-cache"}

# Hacemos la primera petición al endpoint oficial
response = requests.request("GET", url, headers=headers, params=querystring)
res_json = response.json()

print("Respuesta inicial de la AEMET (Paso 1):", res_json)

# --- PASO 2: Extraer la URL fresca y pedir los datos reales ---
if res_json.get("estado") == 200:
    # Ahora sí, 'datos' contendrá una URL fresca y temporal válida para esta sesión
    url_datos_fresca = res_json.get("datos")
    print(f"\n[OK] URL fresca capturada: {url_datos_fresca}")
    
    # Hacemos el segundo GET inmediatamente
    response_final = requests.get(url_datos_fresca)
    
    # ¡Aquí tienes el JSON real con todas las estaciones!
    print("\n--- DATOS FINALES RECUPERADOS (Catálogo de Estaciones) ---")
    print(response_final.text[:1000])  # Mostramos solo los primeros 1000 caracteres para no saturar
else:
    print(f"\n[ERROR] Código {res_json.get('estado')}: {res_json.get('descripcion')}")

In [ ]:
# Intentamos obtener los datos de Málaga capital

In [ ]:
indicativo_estacion = "6172X"  # Málaga
fecha_inicio = "2026-01-01T00:00:00UTC"
fecha_fin = "2026-01-07T23:59:59UTC"

# Endpoint correcto para valores climatológicos diarios
url = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_inicio}/fechafin/{fecha_fin}/estacion/{indicativo_estacion}/"

querystring = {"api_key": TOKEN}
headers = {'cache-control': "no-cache"}

print("Llamando al PASO 1 (Endpoint de Climatología Diaria)...")
response = requests.request("GET", url, headers=headers, params=querystring)
res_json = response.json()

# --- PASO 2: El mismo flujo de antes, pero ahora nos traerá las temperaturas ---
if res_json.get("estado") == 200:
    url_datos_fresca = res_json.get("datos")
    print("[OK] URL fresca obtenida. Descargando datos del rango corto...")
    
    # Petición inmediata a la URL firmada
    response_final = requests.get(url_datos_fresca)
    datos_diarios = response_final.json()
    
    print("\n--- FORMATO DEL PRIMER DÍA RECUPERADO ---")
    if datos_diarios:
        import pprint
        # Imprimimos el primer día completo para que examines las claves de temperatura
        pprint.pprint(datos_diarios[0])
    else:
        print("No hay datos para este rango.")
else:
    print(f"\n[ERROR] Código {res_json.get('estado')}: {res_json.get('descripcion')}")

In [ ]:
df = pd.DataFrame(datos_diarios)

print("--- DATOS BRUTOS ORIGINALES ---")
print(df[['fecha', 'tmed', 'tmax', 'tmin']].head())
print("\nTipos iniciales:\n", df.dtypes)

# ==========================================
# PUERTA DE CALIDAD: FUNCIÓN DE LIMPIEZA DURA
# ==========================================
def limpiar_y_castear_flotante(serie_columna):
    """
    Reemplaza comas por puntos y convierte a float64.
    Si encuentra cualquier cosa que no se pueda transformar, 
    lanza un error en lugar de camuflarlo con NaN.
    """
    # Paso 1: Reemplazar la coma española por el punto anglosajón
    serie_limpia = serie_columna.str.replace(',', '.', regex=False)
    
    # Paso 2: Conversión estricta a numérico (raise lanza error si falla)
    serie_float = pd.to_numeric(serie_limpia, errors='raise')
    
    # Control de calidad intermedio opcional: asegurar que no se nos ha colado ningún nulo oculto
    assert serie_float.notna().all(), f"¡Alerta! Se han generado NaNs en la conversión."
    
    return serie_float

# 2. Aplicamos las transformaciones con red de seguridad
try:
    # Transformamos las temperaturas
    df['tmed'] = limpiar_y_castear_flotante(df['tmed'])
    df['tmax'] = limpiar_y_castear_flotante(df['tmax'])
    df['tmin'] = limpiar_y_castear_flotante(df['tmin'])
    
    # Transformamos la fecha a tipo datetime real
    df['fecha'] = pd.to_datetime(df['fecha'], errors='raise')
    
except Exception as e:
    print(f"\n[💥 ERROR CRÍTICO EN LA PUERTA DE CALIDAD]: {e}")
    raise

# ==========================================
# VERIFICACIÓN FINALES (ASSERTS)
# ==========================================
# Comprobamos que los dtypes se han convertido exactamente a lo que queremos
assert df['tmed'].dtype == 'float64', "tmed no es float64"
assert df['tmax'].dtype == 'float64', "tmax no es float64"
assert df['tmin'].dtype == 'float64', "tmin no es float64"
assert pd.api.types.is_datetime64_any_dtype(df['fecha']), "fecha no es datetime64"

print("\n" + "="*40)
print("¡PUERTA DE CALIDAD SUPERADA CON ÉXITO!")
print("="*40)

# 3. Inspección del resultado final
print("\n--- NUEVOS TYPES DEL DATAFRAME ---")
print(df[['fecha', 'tmed', 'tmax', 'tmin']].dtypes)

print("\n--- ESTADÍSTICAS SENSATAS (describe) ---")
print(df[['tmed', 'tmax', 'tmin']].describe())

In [ ]:
ESTACION_MALAGA = "6172X"

# === 1. PUERTA DE CALIDAD (Limpieza estricta) ===
def limpiar_y_castear_flotante(serie_columna):
    serie_limpia = serie_columna.str.replace(',', '.', regex=False)
    return pd.to_numeric(serie_limpia, errors='raise')


# === 2. EXTRACTOR MENSUAL (Estructura interna validada) ===
def extraer_historico_aemet(indicativo_estacion, anio_objetivo):
    inicios_mes = pd.date_range(start=f"{anio_objetivo}-01-01", end=f"{anio_objetivo}-12-01", freq="MS")
    finales_mes = pd.date_range(start=f"{anio_objetivo}-01-01", end=f"{anio_objetivo}-12-31", freq="ME")
    intervalos = list(zip(inicios_mes, finales_mes))
    
    dataframes_validos = []
    meses_en_cuarentena = []
    
    headers = {'cache-control': "no-cache"}
    querystring = {"api_key": TOKEN}
    
    for inicio, fin in intervalos:
        mes_actual_str = inicio.strftime('%Y-%m')
        
        # Evitar pedir meses futuros (estamos en julio de 2026)
        if inicio > pd.Timestamp.now():
            continue
            
        fecha_ini_str = inicio.strftime("%Y-%m-%dT00:00:00UTC")
        fecha_fin_str = fin.strftime("%Y-%m-%dT23:59:59UTC")
        
        url_paso1 = f"https://opendata.aemet.es/opendata/api/valores/climatologicos/diarios/datos/fechaini/{fecha_ini_str}/fechafin/{fecha_fin_str}/estacion/{indicativo_estacion}/"
        
        exito_mes = False
        intentos_totales = 0  
        MAX_INTENTOS = 4     
        
        while not exito_mes and intentos_totales < MAX_INTENTOS:
            intentos_totales += 1
            res_paso1 = requests.request("GET", url_paso1, headers=headers, params=querystring)
            json_paso1 = res_paso1.json()
            
            if json_paso1.get("estado") == 200:
                url_fresca = json_paso1.get("datos")
                res_paso2 = requests.get(url_fresca)
                
                if res_paso2.status_code != 200:
                    time.sleep(10)
                    continue
                
                try:
                    datos_mes = res_paso2.json()
                except Exception:
                    time.sleep(10)
                    continue
                
                try:
                    df_mes = pd.DataFrame(datos_mes)
                    
                    if df_mes.empty:
                        raise ValueError("El servidor devolvió una estructura vacía.")
                    
                    # -----------------------------------------------------
                    # 🔧 CORRECCIÓN DEL BUG: Ahora sí sobreescribimos tmin
                    # -----------------------------------------------------
                    df_mes['tmed'] = limpiar_y_castear_flotante(df_mes['tmed'])
                    df_mes['tmax'] = limpiar_y_castear_flotante(df_mes['tmax'])
                    df_mes['tmin'] = limpiar_y_castear_flotante(df_mes['tmin'])  # <- ¡Arreglado!
                    df_mes['fecha'] = pd.to_datetime(df_mes['fecha'], errors='raise')
                    
                    dataframes_validos.append(df_mes)
                    print(f" -> [OK] {mes_actual_str} extraído ({len(df_mes)} días).")
                    exito_mes = True 
                    
                except Exception as e:
                    motivo_error = f"Error en capa de datos/parseo: {str(e)}"
                    print(f" -> [🚨 CUARENTENA] {mes_actual_str} aislado. Motivo: {e}")
                    meses_en_cuarentena.append({"mes": mes_actual_str, "error": motivo_error})
                    exito_mes = True 
                
            elif json_paso1.get("estado") == 429:
                print(f" -> [💥 RATE LIMIT] 429 en {mes_actual_str}. Durmiendo 60 segundos...")
                time.sleep(60) 
            else:
                motivo_error = f"Error API Paso 1 ({json_paso1.get('estado')}): {json_paso1.get('descripcion')}"
                print(f" -> [🚨 CUARENTENA] {mes_actual_str} aislado. Motivo: {json_paso1.get('descripcion')}")
                meses_en_cuarentena.append({"mes": mes_actual_str, "error": motivo_error})
                exito_mes = True
        
        if not exito_mes:
            meses_en_cuarentena.append({"mes": mes_actual_str, "error": "Agotados intentos de red."})
            
        time.sleep(2) 
        
    return dataframes_validos, meses_en_cuarentena


# === 3. BUCLE MULTI-AÑO (2023 - 2026) ===
todos_los_dataframes = []
cuarentena_global = []
años_objetivo = [2023, 2024, 2025, 2026]

print(f"🚀 INICIANDO EXTRACCIÓN HISTÓRICA COMPLETA (2023-2026)")
print("=" * 60)

for anio in años_objetivo:
    dfs_anio, cuarentena_anio = extraer_historico_aemet(ESTACION_MALAGA, anio)
    
    if dfs_anio:
        todos_los_dataframes.extend(dfs_anio)
    if cuarentena_anio:
        cuarentena_global.extend(cuarentena_anio)

# === 4. ENSAMBLADO Y REPORTE FINAL ===
print("\n" + "=" * 60)
print("🏁 FIN DEL PROCESAMIENTO GLOBAL")
print("=" * 60)  # 🔧 CORRECCIÓN COSMÉTICA: Ya no imprime "=50"

if todos_los_dataframes:
    df_historico_completo = pd.concat(todos_los_dataframes, ignore_index=True)
    
    print(f"\n✅ Extracción terminada. Registros totales salvados: {len(df_historico_completo)} días.")
    print(f"❌ Bloques mensuales enviados a cuarentena: {len(cuarentena_global)}")
    
    # 🔧 CORRECCIÓN EXTRA: Incluimos 'tmin' en el describe() para verificar el casteo
    print("\n📊 RESUMEN ESTADÍSTICO (DE LAS TRES TEMPERATURAS):")
    print(df_historico_completo[['tmed', 'tmax', 'tmin']].describe())
else:
    print("\n🚨 Error crítico: No se ha podido salvar ningún dato del periodo.")

if cuarentena_global:
    print("\n📋 REGISTRO DETALLADO DE LA CUARENTENA:")
    for item in cuarentena_global:
        print(f"  • {item['mes']}: {item['error']}")
else:
    print("\n🎉 ¡Milagro! Ningún mes del histórico requirió cuarentena.")

In [ ]:
from src.aemet_client import (
    limpiar_y_castear_flotante,
    extraer_historico_aemet_estacion,
    extraer_historico_multi_estacion,
)


In [ ]:
# Definición de variables
ESTACIONES_INTERES = [
    "3129",   # Madrid Aeropuerto
    "0076",   # Barcelona Aeropuerto
    "8414A",  # Valencia Aeropuerto
    "9434",   # Zaragoza Aeropuerto
    "5783",   # Sevilla Aeropuerto
    "6155A",  # Málaga Aeropuerto
    "1082",   # Bilbao Aeropuerto
    "2539"    # Valladolid Aeropuerto
]
# Llamada a la función
df_clima, cuarentena = extraer_historico_multi_estacion(
    indicativos=ESTACIONES_INTERES,
    fecha_inicio="2023-01-01",
    fecha_fin="2023-01-31",
    token=TOKEN
)

In [ ]:
df_clima

In [ ]:
df_clima.info()
df_clima["indicativo"].unique()

In [ ]:
def interpolar_temperaturas(df, limite_hueco=3):
    """
    Ordena por estación y fecha, asegura el formato numérico de tmin y tmax 
    sin alterar el indicativo alfanumérico, e interpola con límite de días.
    """
    # 1. Copia profunda para no alterar tu df original
    df_temp = df.copy()
    
    # 2. Aseguramos formato de fecha y ordenamos por estación (indicativo) y cronología
    df_temp['fecha'] = pd.to_datetime(df_temp['fecha'])
    df_temp = df_temp.sort_values(by=['indicativo', 'fecha']).reset_index(drop=True)
    
    # 3. Conversión segura de tmin y tmax (solo si no son float64 ya)
    for col in ['tmin', 'tmax']:
        if col in df_temp.columns:
            # Si Pandas lo detectó como texto/objeto, limpiamos comas
            if df_temp[col].dtype == 'object':
                df_temp[col] = df_temp[col].astype(str).str.replace(',', '.', regex=False)
            
            # Convertimos a numérico de forma segura; los errores o textos extraños serán NaN
            df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
            
    # 4. Interpolación lineal agrupada por la estación ('indicativo')
    for col in ['tmin', 'tmax']:
        if col in df_temp.columns:
            # Agrupamos por indicativo para no mezclar estaciones y aplicamos la interpolación
            df_temp[col] = df_temp.groupby('indicativo')[col].transform(
                lambda x: x.interpolate(method='linear', limit=limite_hueco, limit_area='inside')
            )
            
    return df_temp

In [ ]:
df_interpolado = interpolar_temperaturas(df_clima)
df_interpolado

In [ ]:
df_interpolado["indicativo"].unique()

In [ ]:
df_interpolado.loc[df_interpolado["indicativo"]=="9434"]

In [ ]:
antes = df_clima.groupby('indicativo')[['tmin','tmax']].apply(lambda x: x.isna().sum())
despues = df_interpolado.groupby('indicativo')[['tmin','tmax']].apply(lambda x: x.isna().sum())
print("NaN antes:\n", antes, "\nNaN después:\n", despues)

In [ ]:
df_interpolado.loc[df_interpolado.indicativo=="2539", ["fecha","tmin","tmax"]].iloc[26:31]

In [ ]:
ZONAS = {
    "3129": "continental", "9434": "continental", "2539": "continental",
    "0076": "mediterraneo", "8414A": "mediterraneo", "6155A": "mediterraneo",
    "5783": "guadalquivir",
    "1082": "cantabrico",
}
df_interpolado["zona"] = df_interpolado["indicativo"].map(ZONAS)

In [ ]:
assert df_interpolado["zona"].notna().all(), "Hay estación sin zona (revisa claves del dict)"


In [ ]:
PESOS = {
    "3129": 0.401,    # Madrid
    "9434": 0.081,    # Zaragoza
    "2539": 0.035,    # Valladolid
    "0076": 0.198,    # Barcelona
    "8414A": 0.097,   # Valencia
    "6155A": 0.069,   # Málaga
    "5783": 0.079,    # Sevilla
    "1082": 0.040     # Bilbao
}

In [ ]:
# Añadir columnas a df_interpolado
df_interpolado["zona"] = df_interpolado["indicativo"].map(ZONAS)
df_interpolado["peso"] = df_interpolado["indicativo"].map(PESOS)

# Verificación de que no hay NaN en la columna peso
assert df_interpolado["peso"].notna().all(), "¡Alerta! Hay indicativos en el DataFrame que no tienen un peso asignado."

In [ ]:
def media_ponderada(df_grupo, col_valor, col_peso):
    # Nos quedamos solo con las filas que tienen valor y peso válido
    validos = df_grupo[df_grupo[col_valor].notna() & df_grupo[col_peso].notna()]
    if validos.empty:
        return np.nan
    
    # Suma(valor * peso) / Suma(peso)
    return (validos[col_valor] * validos[col_peso]).sum() / validos[col_peso].sum()

# Agrupar por fecha y zona calculando tmin y tmax ponderados
df_agrupado = df_interpolado.groupby(["fecha", "zona"]).apply(
    lambda g: pd.Series({
        "tmin": media_ponderada(g, "tmin", "peso"),
        "tmax": media_ponderada(g, "tmax", "peso")
    })
).reset_index()

# Pivotar a formato ancho
df_ancho = df_agrupado.pivot(index="fecha", columns="zona", values=["tmin", "tmax"])

# Colapsar el multi-índice de las columnas: (tmin, continental) -> tmin_continental
df_ancho.columns = [f"{variable}_{zona}" for variable, zona in df_ancho.columns]
df_final = df_ancho.reset_index()

In [ ]:
# Elegimos una fecha de prueba
fecha_test = df_final["fecha"].iloc[0]

# Tmin de Sevilla (5783) cruda vs final
tmin_sevilla_cruda = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "5783")]["tmin"].values[0]
tmin_sevilla_final = df_final[df_final["fecha"] == fecha_test]["tmin_guadalquivir"].values[0]

# Tmin de Bilbao (1082) cruda vs final
tmin_bilbao_cruda = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "1082")]["tmin"].values[0]
tmin_bilbao_final = df_final[df_final["fecha"] == fecha_test]["tmin_cantabrico"].values[0]

print(f"¿Coincide Sevilla? Cruda: {tmin_sevilla_cruda} | Final: {tmin_sevilla_final}")
print(f"¿Coincide Bilbao?  Cruda: {tmin_bilbao_cruda}  | Final: {tmin_bilbao_final}")

assert np.isclose(tmin_sevilla_cruda, tmin_sevilla_final), "Fallo en verificación de Sevilla"
assert np.isclose(tmin_bilbao_cruda, tmin_bilbao_final), "Fallo en verificación de Bilbao"
print("¡Verificación 1 SUPERADA exitosamente!")

In [ ]:
# Elegimos una fecha de prueba cualquiera
fecha_test = df_final["fecha"].iloc[0]

# Obtenemos los valores individuales de tmin para esa fecha
tmin_mad = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "3129")]["tmin"].values[0]
tmin_zar = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "9434")]["tmin"].values[0]
tmin_val = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "2539")]["tmin"].values[0]

# Cálculo manual usando pesos
num = (tmin_mad * 0.401) + (tmin_zar * 0.081) + (tmin_val * 0.035)
den = 0.401 + 0.081 + 0.035
tmin_cont_manual = num / den

# Valor obtenido en df_final
tmin_cont_final = df_final[df_final["fecha"] == fecha_test]["tmin_continental"].values[0]

print(f"Tmin Continental Manual: {tmin_cont_manual:.4f}")
print(f"Tmin Continental Final : {tmin_cont_final:.4f}")

assert np.isclose(tmin_cont_manual, tmin_cont_final), "La media ponderada continental no coincide."
print("¡Verificación 2 SUPERADA exitosamente!")

In [ ]:
# 1. Creamos una copia del dataset original para simular el fallo de red/estación
df_simulado = df_interpolado.copy()

# 2. Borramos la fila de Valladolid (2539) en la primera fecha de la serie
fecha_test = df_simulado["fecha"].iloc[0]
df_simulado = df_simulado[~((df_simulado["fecha"] == fecha_test) & (df_simulado["indicativo"] == "2539"))]

# 3. Recalculamos la agrupación en el dataset simulado
df_agrupado_sim = df_simulado.groupby(["fecha", "zona"]).apply(
    lambda g: pd.Series({
        "tmin": media_ponderada(g, "tmin", "peso"),
        "tmax": media_ponderada(g, "tmax", "peso")
    })
).reset_index()
df_final_sim = df_agrupado_sim.pivot(index="fecha", columns="zona", values=["tmin", "tmax"])
df_final_sim.columns = [f"{v}_{z}" for v, z in df_final_sim.columns]
df_final_sim = df_final_sim.reset_index()

# 4. Cálculo esperado a mano sin Valladolid: (Madrid + Zaragoza) / (0.401 + 0.081)
tmin_mad = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "3129")]["tmin"].values[0]
tmin_zar = df_interpolado[(df_interpolado["fecha"] == fecha_test) & (df_interpolado["indicativo"] == "9434")]["tmin"].values[0]

num_faltante = (tmin_mad * 0.401) + (tmin_zar * 0.081)
den_faltante = 0.401 + 0.081
tmin_cont_faltante_manual = num_faltante / den_faltante

tmin_cont_faltante_final = df_final_sim[df_final_sim["fecha"] == fecha_test]["tmin_continental"].values[0]

print(f"Faltando Valladolid - Cálculo Manual: {tmin_cont_faltante_manual:.4f}")
print(f"Faltando Valladolid - Cálculo Final : {tmin_cont_faltante_final:.4f}")

assert np.isclose(tmin_cont_faltante_manual, tmin_cont_faltante_final), "El denominador no se recalculó correctamente tras omitir la estación."
print("¡Verificación 3 SUPERADA exitosamente! El denominador se encoge de forma orgánica.")

In [ ]:
df_final

In [ ]:
assert df_final["fecha"].is_unique
assert df_final.shape[1] == 9  # fecha + 8 zonas×variable

Extracción de todo

In [ ]:
# Definición de variables
ESTACIONES_INTERES = [
    "3129",   # Madrid Aeropuerto
    "0076",   # Barcelona Aeropuerto
    "8414A",  # Valencia Aeropuerto
    "9434",   # Zaragoza Aeropuerto
    "5783",   # Sevilla Aeropuerto
    "6155A",  # Málaga Aeropuerto
    "1082",   # Bilbao Aeropuerto
    "2539"    # Valladolid Aeropuerto
]
df_clima, cuarentena = extraer_historico_multi_estacion(
    indicativos=ESTACIONES_INTERES,
    fecha_inicio="2023-01-01",
    fecha_fin="2026-07-17",   # hoy — todo lo posterior irá directo a cuarentena
    token=TOKEN
)

In [ ]:
cuarentena

In [ ]:
# --- Gestión preventiva de columnas de texto/hora con NaNs para PyArrow ---
columnas_horarias = [
    "horatmin",
    "horatmax",
    "horaracha",
    "horaPresMax",
    "horaPresMin",
    "horaHrMax",
    "horaHrMin",
    "horaPIntMax",
]
for col in columnas_horarias:
    if col in df_clima.columns:
        # Forzamos el uso del tipo String nativo de pandas que maneja nulos de forma segura
        df_clima[col] = df_clima[col].astype(pd.StringDtype())

# 1. Guardar df_clima a Parquet usando PyArrow
df_clima.to_parquet(PATH_AEMET_CRUDO, engine="pyarrow", index=False)
print(f"✔ DataFrame df_clima guardado con éxito en: {PATH_AEMET_CRUDO}")

# 2. Guardar objeto cuarentena real a JSON
cuarentena_real = cuarentena.copy()

with open(PATH_CUARENTENA, "w", encoding="utf-8") as f:
    json.dump(cuarentena_real, f, ensure_ascii=False, indent=4)
print(f"✔ Cuarentena real persistida con éxito en: {PATH_CUARENTENA}")


In [ ]:
# 1. Releer el archivo Parquet recién creado
df_clima_releido = pd.read_parquet(PATH_AEMET_CRUDO, engine="pyarrow")

# --- VERIFICACIÓN A: Shape idéntico ---
assert (
    df_clima_releido.shape == df_clima.shape
), f"¡Fallo en las dimensiones! Original: {df_clima.shape} | Releído: {df_clima_releido.shape}"
print(f"✔ Verificación de dimensiones superada: {df_clima_releido.shape}")

# --- VERIFICACIÓN B: Tipos numéricos intactos (Float puro) ---
for col in ["tmin", "tmax", "tmed"]:
    if col in df_clima_releido.columns:
        assert pd.api.types.is_float_dtype(
            df_clima_releido[col]
        ), f"¡Peligro! La columna '{col}' ha regresado rota como tipo: {df_clima_releido[col].dtype}"
print(
    "✔ Verificación de tipos superada: 'tmin', 'tmax' y 'tmed' se mantienen como floats numéricos."
)

# --- VERIFICACIÓN C: Conteo idéntico por indicativo ---
conteo_original = df_clima["indicativo"].value_counts().sort_index()
conteo_releido = df_clima_releido["indicativo"].value_counts().sort_index()

pd.testing.assert_series_equal(
    conteo_original, conteo_releido, check_names=False
)
print(
    "✔ Verificación de frecuencias superada: Los conteos por indicativo coinciden exactamente."
)

print(
    "\n🎉 ¡ROUND-TRIP COMPLETADO EXITOSAMENTE! Persistencia y tipos validados directamente contra df_clima."
)

In [ ]:
print(df_clima["prec"].isna().sum())
print(df_clima_releido["prec"].isna().sum())